# Tokenizer Fragmentation Analysis

Analyzes how tokenizer behavior differs between LLaMA-2 (English) and PersianLLaMA (Persian).

Key questions:
- Does Persian text get fragmented into more subwords than English?
- Is fragmentation correlated with bias scores?
- Which bias categories show the highest fragmentation?

In [ ]:
!pip install transformers accelerate -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from transformers import AutoTokenizer
sns.set_theme(style='whitegrid')

In [ ]:
!huggingface-cli login

In [ ]:
BASE = '/content/drive/MyDrive/'
df = pd.read_excel(BASE + 'NewStreoSet_Dataset.xlsx')
pll_df = pd.read_csv(BASE + 'stereoset_LLama_ByBiasType.csv')
print(df.shape, pll_df.shape)

In [ ]:
# Load both tokenizers
tok_en = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-hf')
tok_fa = AutoTokenizer.from_pretrained('Narabzad/PersianLLaMA-13B')

In [ ]:
def fragmentation_ratio(sentence, tokenizer):
 """Subword tokens / whitespace words"""
 tokens = tokenizer.tokenize(str(sentence))
 words = str(sentence).split()
 return len(tokens) / max(len(words), 1)

df['token_count_en'] = df['full_sentence'].apply(lambda s: len(tok_en.tokenize(str(s))))
df['token_count_fa'] = df['Persian'].apply(lambda s: len(tok_fa.tokenize(str(s))))
df['word_count_en'] = df['full_sentence'].apply(lambda s: len(str(s).split()))
df['word_count_fa'] = df['Persian'].apply(lambda s: len(str(s).split()))
df['frag_ratio_en'] = df['token_count_en'] / df['word_count_en']
df['frag_ratio_fa'] = df['token_count_fa'] / df['word_count_fa']
print(df[['frag_ratio_en','frag_ratio_fa']].describe().round(3))

In [ ]:
# Distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df['frag_ratio_en'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('English Fragmentation Ratio (LLaMA-2)')
axes[0].set_xlabel('Tokens per Word')
sns.histplot(df['frag_ratio_fa'], kde=True, ax=axes[1], color='coral')
axes[1].set_title('Persian Fragmentation Ratio (PersianLLaMA)')
axes[1].set_xlabel('Tokens per Word')
plt.tight_layout()
plt.savefig('fragmentation_distribution.png', dpi=150)
plt.show()
print(f"Mean EN frag: {df['frag_ratio_en'].mean():.3f}")
print(f"Mean FA frag: {df['frag_ratio_fa'].mean():.3f}")

In [ ]:
# Fragmentation by bias type
frag_by_type = df.groupby('bias_type')[['frag_ratio_en', 'frag_ratio_fa']].mean().round(3)
print('Fragmentation by bias type:')
print(frag_by_type)

frag_by_type.plot(kind='bar', figsize=(10,6), color=['steelblue','coral'])
plt.title('Fragmentation Ratio by Bias Type')
plt.ylabel('Tokens per Word')
plt.xlabel('Bias Type')
plt.xticks(rotation=0)
plt.legend(['English (LLaMA-2)', 'Persian (PersianLLaMA)'])
plt.tight_layout()
plt.savefig('fragmentation_by_type.png', dpi=150)
plt.show()

In [ ]:
# Save extended dataset with tokenizer features
df.to_csv(BASE + 'stereoset_with_tokenizer_features.csv', index=False)
print('Saved stereoset_with_tokenizer_features.csv')